In [1]:
import requests
from pathlib import Path
import pandas as pd
from datetime import datetime
from openai import OpenAI
import json
import time
import math
import numpy as np
from IPython.display import display, Markdown, Latex
import re
from IPython.display import Image, display
import cv2
import os
import matplotlib.pyplot as plt
import base64
from bs4 import BeautifulSoup

C:\Users\mariu\basic_env_3_14\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
try:
    with open(r"klucz_API_AIDEVS.txt", "r") as f:
        api_key = f.read().strip()  # Usuwa ewentualne znaki nowej linii
except FileNotFoundError:
    print("Błąd: Plik klucza API nie został odnaleziony.")
except Exception as e:
    print(f"Wystąpił błąd podczas odczytu pliku: {e}")

try:
    with open(r"klucz_API_openrouter.txt", "r") as f:
        api_key_openrouter = f.read().strip()  # Usuwa ewentualne znaki nowej linii
except FileNotFoundError:
    print("Błąd: Plik klucza API nie został odnaleziony.")
except Exception as e:
    print(f"Wystąpił błąd podczas odczytu pliku: {e}")

In [6]:
#url_toolsearch = "https://hub.ag3nts.org/api/toolsearch"
url_toolsearch = "https://hub.ag3nts.org/api"
url_api = "https://hub.ag3nts.org/verify"
url_preview = "https://hub.ag3nts.org/savethem_preview.html"

In [90]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key_openrouter # Zmienna wczytana wcześniej z pliku klucza
)

In [5]:
def api_action(endpoint,query):    
    headers = {
        "Content-Type": "application/json"
    }
    
    payload = {
        "apikey": api_key,
        "query": query,
    }

    response = requests.post(f"{url_toolsearch}/{endpoint}", data=json.dumps(payload), headers=headers)
    return {'status_code': response.status_code, 'message': response.json()}

In [8]:
api_action("toolsearch","Provide information about map")

{'status_code': 200,
 'message': {'code': 210,
  'message': 'Matching tools found.',
  'query': 'Provide information about map',
  'tools': [{'name': 'maps',
    'url': '/api/maps',
    'description': 'Terrain maps and map-related location resources.',
    'parameter': 'query',
    'score': 5,
    'matched_keywords': ['map']},
   {'name': 'wehicles',
    'url': '/api/wehicles',
    'description': 'Information about vehicles and means of transportation.',
    'parameter': 'query',
    'score': 1,
    'matched_keywords': ['information']}]}}

In [79]:
def init_gathered_tools():
    global gathered_tools
    gathered_tools=dict()

In [81]:
def gather_tool_information(query):
    global gathered_tools
    resp = api_action("toolsearch",query)
    if resp['message']['code'] == 210:
        tools_found = resp['message']['tools']
        tools_found1 = {x['name']: x['description'] for x in tools_found}
        gathered_tools = gathered_tools | tools_found1
        return
    

In [82]:
gather_tool_information("Provide information about map")

In [83]:
gathered_tools

{'maps': 'Terrain maps and map-related location resources.',
 'wehicles': 'Information about vehicles and means of transportation.'}

In [73]:
gather_tool_information("toolsearch")

[]


In [84]:
def print_tool_info():
    str_out = '\n'.join([f'{k}: {x}' for k,x in gathered_tools.items()])
    return str_out

In [85]:
print(print_tool_info())

maps: Terrain maps and map-related location resources.
wehicles: Information about vehicles and means of transportation.


In [103]:
SYSTEM_PROMPT_DISCOVER = """\
You are a reconnaissance agent. Your ONLY job is to discover and catalog available tools by calling gather_tool_information(query) with well-chosen queries.

You are preparing intelligence for a SEPARATE planning agent that will later solve this task:
- Guide a messenger to the city of Skolwin on a 10x10 grid map
- Choose a vehicle (or go on foot)
- Manage 10 food units and 10 fuel units
- Balance speed (fuel cost) vs travel time (food cost)
- Submit optimal route as directions: ["vehicle_name", "right", "up", ...]

## YOUR TOOLS
gather_tool_information(query: str) -> None
This function:
- Calls toolsearch API with your query
- Stores discovered tools in the global `gathered_tools` dict {name: description}
- Returns nothing — results accumulate silently

print_tool_info() -> str
This function:
- Returns contents of `gathered_tools` dict as string containing lines: 'name: description' where each line refers to separate discovered API endpoint. 

## YOUR MISSION
Systematically explore ALL relevant tool categories by issuing diverse queries. Cover every domain the planning agent will need. Do NOT plan the route — only gather tools.

## QUERY STRATEGY
Issue queries across ALL of these domains (use varied phrasing, not just keywords):

### Map & Terrain
- "get map"
- "terrain information grid"
- "map obstacles rivers trees stones"
- "passable fields movement restrictions"
- "notes on map"

### Vehicles
- "available vehicles list"
- "vehicle speed fuel consumption"
- "transport options messenger"
- "vehicle parameters stats"

### Movement & Rules
- "movement rules directions"
- "movement cost per step"
- "on foot walking speed food consumption"
- "exit vehicle continue on foot"
- "walk"
- "dismount"

### Resources
- "food consumption per move"
- "fuel usage speed relationship"
- "resource management food fuel"


## EXECUTION RULES
1. Call gather_tool_information() at least 15–20 times with VARIED queries
2. Never repeat the same query twice
3. Use both natural language AND keyword-style queries for the same topic (toolsearch handles both)
4. If a tool name hints at functionality you haven't queried yet, probe that area immediately
5. Stop only when gathered_tools has been stable (no new tools added) for 5+ consecutive queries

## OUTPUT
When finished, print the complete gathered_tools dictionary and a brief summary of what each tool likely does. Do NOT attempt to plan a route or call any other function.
"""

In [104]:
tools_discover_agent = [
    {
        "type": "function",
        "function": {
            "name": "gather_tool_information",
            "description": "Performs searches on available tools and adds them to Python dictionary of available tools.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                      "type": "string",
                      "description": "Search query. Include keywords which can help find needed tools",
                    },
                },
                    "required": ["query"],
                  "additionalProperties": False
              },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "print_tool_info",
            "description": "Prints gathered information on available tools with `gather_tool_information` so far.",
            "parameters": {
                "type": "object",
                "properties": {
                },
                    "required": [],
                  "additionalProperties": False
              },
        }
    },
]

In [105]:
tools_discover_agent_map = {
    "gather_tool_information": gather_tool_information,
    "print_tool_info": print_tool_info,
}

In [106]:
def agent_loop(agent_name,sys_prompt, tools, tools_map, max_iterations):
    messages = [{"role": "system", "content": sys_prompt}]
    iteration = 0
    
    print(f"=== Starting {agent_name} ===")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        
        completion = client.chat.completions.create(
            model="openai/gpt-5-mini", 
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
    
        response_message = completion.choices[0].message
        print('Response message')
        print(response_message)
        #messages.append(response_message)
        messages.append({
            "role": response_message.role,
            "content": response_message.content,
            "tool_calls": response_message.tool_calls
        })
        
        if not response_message.tool_calls:
            print("Model finished without tools. Trying to parse result.")
            break
            
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            if function_name in tools_map:
                try:
                    result = tools_map[function_name](**arguments)
                    w = {
                        "role": "tool",
                        "name": function_name,
                        "content": json.dumps(result),
                        "tool_call_id": tool_call.id
                    }
                    print(w)
                    messages.append(w)
                except Exception as e:
                    print(f"Error in {function_name}: {e}")
    
    print(f"=== {agent_name} Finished ===")
    return messages

In [107]:
init_gathered_tools()
messages = agent_loop("Discover Agent",SYSTEM_PROMPT_DISCOVER,tools_discover_agent,tools_discover_agent_map,50)

=== Starting Discover Agent ===

--- Iteration 1 ---
Response message
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d6qTRgqFjkiYm31KD5oQeRFf', function=Function(arguments='{"query":"get map"}', name='gather_tool_information'), type='function', index=0)], reasoning="**Gathering tool information**\n\nI need to call gather_tool_information about 15-20 times with different queries, avoiding repeats and using both natural language and keywords. It’s essential to explore all functionality hinted by tool names while monitoring the responses. Once I see stability in returned content for five consecutive queries, I'll stop. The function is in the functions namespace, so I'll directly call functions.gather_tool_information instead of using multi_tool_use.parallel. I’ll prepare varied queries related to Map & Terrain, Vehicles, Movement & Rules, and Resources.**Planni

In [108]:
gathered_tools

{'maps': 'Terrain maps and map-related location resources.',
 'wehicles': 'Information about vehicles and means of transportation.',
 'books': 'Old books and notebooks with various notes.'}

In [137]:
def init_gathered_notes():
    global gathered_notes
    gathered_notes=dict()

def gather_notes(query):
    global gathered_notes
    resp = api_action("books",query)
    if resp['message']['code'] == 220:
        notes_found = resp['message']['notes']
        notes_found1 = {x['id']: {'content': x['content'], 'title':x['title']} for x in notes_found}
        gathered_notes = gathered_notes | notes_found1
        return

def print_notes_info():
    str_out = '\n'.join([f'{k}: {x["title"]} - {x["content"]}' for k,x in gathered_notes.items()])
    return str_out

In [138]:
tools_discover2_agent = [
    {
        "type": "function",
        "function": {
            "name": "gather_notes",
            "description": "Performs searches on available notes and adds them to Python dictionary of available notes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                      "type": "string",
                      "description": "Search query. Include keywords which can help find new notes",
                    },
                },
                    "required": ["query"],
                  "additionalProperties": False
              },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "print_notes_info",
            "description": "Prints gathered information on available notes with `gather_notes` so far.",
            "parameters": {
                "type": "object",
                "properties": {
                },
                    "required": [],
                  "additionalProperties": False
              },
        }
    },
]

In [139]:
tools_discover2_agent_map = {
    "gather_notes": gather_notes,
    "print_notes_info": print_notes_info,
}

In [127]:
SYSTEM_PROMPT_DISCOVER2 = """\
You are a knowledge extraction agent. Your ONLY job is to discover and catalog all relevant notes by calling gather_notes(query) with well-chosen queries.

You are preparing knowledge for a SEPARATE planning agent that will solve this task:
- Guide a messenger to the city of Skolwin on a 10x10 grid map
- Choose optimal vehicle given fuel/food constraints
- Submit route as: ["vehicle_name", "right", "up", ...]

The notes endpoint ("books") contains lore, rules, and data entries about the world — terrain descriptions, vehicle specs, movement rules, resource costs, map data, etc.

## YOUR TOOLS
gather_notes(query: str) -> None
- Queries the "books" endpoint with your query string
- Stores discovered notes in global `gathered_notes` dict: {id: {title, content}}
- Returns nothing — results accumulate silently
- Always returns up to 3 best-matching notes per query

print_notes_info() -> str
- Returns a formatted string of all currently gathered notes
- Call this periodically to review what has been collected so far

## YOUR MISSION
Systematically query ALL relevant knowledge domains using diverse queries. The planning agent will need complete information — missing a key note (e.g. about fuel costs or terrain passability) could make the route unsolvable.

## QUERY STRATEGY
Issue queries across ALL of these domains. Use both natural language and keyword variants:

### Map & Terrain
- "map layout Skolwin"
- "terrain types grid fields"
- "river forest stone obstacle passable"
- "10x10 map description"
- "starting position base location"
- "Skolwin city destination coordinates"

### Vehicles
- "available vehicles"
- "vehicle speed statistics"
- "fuel consumption per move"
- "vehicle list parameters"
- "transport options horse cart"

### Movement Rules
- "movement rules directions"
- "movement cost step"
- "walking on foot speed"
- "exit vehicle continue on foot rules"
- "how movement works"

### Resources & Costs
- "food consumption travel"
- "fuel usage speed"
- "resource cost per field"
- "food fuel balance"
- "provisions messenger"

### Route & Navigation
- "optimal path rules"
- "how to reach Skolwin"
- "navigation instructions"
- "route planning constraints"

### General Lore / Meta
- "mission briefing"
- "messenger task description"
- "savethem task rules"
- "game rules overview"
- "notes introduction"

## EXECUTION RULES
1. Call gather_notes() at least 20 times with VARIED queries
2. Never repeat the exact same query
3. After every 5 calls, invoke print_notes_info() and review collected titles
4. If a note title suggests an unexplored topic, immediately probe that topic with follow-up queries
5. If two consecutive query batches yield zero NEW note IDs, try completely different keywords before stopping
6. Stop only when gathered_notes has been stable (no new IDs added) for 6+ consecutive queries

## OUTPUT
When done:
1. Call print_notes_info() one final time
2. Print a structured summary grouped by topic:
   - Map data (terrain, positions)
   - Vehicle specs (names, speed, fuel cost, food cost)
   - Movement rules
   - Resource constraints
   - Any other relevant findings
3. Highlight any GAPS — topics you queried but found no notes for

Do NOT plan a route. Do NOT call any other functions. Only gather and summarize.
"""

In [140]:
init_gathered_notes()
messages2 = agent_loop("Discover Agent 2",SYSTEM_PROMPT_DISCOVER2,tools_discover2_agent,tools_discover2_agent_map,100)

=== Starting Discover Agent 2 ===

--- Iteration 1 ---
Response message
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_MhAL3a5lkjt2WrBorhBbwnS0', function=Function(arguments='{"query":"map layout Skolwin"}', name='gather_notes'), type='function', index=0)], reasoning="**Finalizing notes and execution**\n\nI’m thinking about how we need to stop only after achieving stability for six or more consecutive queries, but the challenge is I can’t detect stability. After processing numerous queries, I believe I've gathered most of the notes, so it’s time to proceed with the final print and summary. I’ll implement the function calls one by one, using functions.gather_notes and print_notes_info after every five. Let’s go ahead and execute this!", reasoning_details=[{'type': 'reasoning.summary', 'summary': "**Finalizing notes and execution**\n\nI’m thinking about how we

In [141]:
gathered_notes

{'maps-endpoint': {'content': 'Terrain maps for any city can be obtained from the /api/maps endpoint. To get the correct map, send the name of the city you want to reach in the request. This makes the maps service the standard source of terrain data when planning a mission toward a specific destination. If someone needs a fresh layout for route analysis, that endpoint is where to ask first.',
  'title': 'Getting terrain maps for target cities'},
 'legend-markers': {'content': 'The map uses a compact set of symbols, and every letter matters during route planning. The letter T marks a tree tile, W marks water, and R marks rocks that block movement completely. The letter S marks the starting position, while G marks the goal that ends the mission successfully. If someone misreads even one marker, the whole path can collapse before the traveler reaches the city. Treat the legend as the first thing to decode before testing any route.',
  'title': 'Map legend and terrain markers'},
 'no-gas-s

In [143]:
mapa = api_action("maps","Skolwin")

In [144]:
mapa

{'status_code': 200,
 'message': {'code': 241,
  'message': 'Map found.',
  'cityName': 'Skolwin',
  'map': [['.', '.', '.', '.', '.', '.', '.', '.', 'W', 'W'],
   ['.', '.', '.', '.', '.', '.', '.', 'W', 'W', '.'],
   ['.', 'T', '.', '.', '.', '.', 'W', 'W', '.', '.'],
   ['.', '.', '.', '.', '.', '.', 'W', '.', '.', '.'],
   ['.', '.', 'T', '.', '.', '.', 'W', '.', 'G', '.'],
   ['.', '.', '.', '.', 'R', '.', 'W', '.', '.', '.'],
   ['.', '.', '.', 'R', 'R', '.', 'W', 'W', '.', '.'],
   ['S', 'R', '.', '.', '.', '.', '.', 'W', '.', '.'],
   ['.', '.', '.', '.', '.', '.', 'W', 'W', '.', '.'],
   ['.', '.', '.', '.', '.', 'W', 'W', '.', '.', '.']],
  'text': '........WW\n.......WW.\n.T....WW..\n......W...\n..T...W.G.\n....R.W...\n...RR.WW..\nSR.....W..\n......WW..\n.....WW...'}}

In [148]:
print(mapa['message']['text'])

........WW
.......WW.
.T....WW..
......W...
..T...W.G.
....R.W...
...RR.WW..
SR.....W..
......WW..
.....WW...


In [154]:


vehicle_notes = [{'name':x,
                  'message': api_action("wehicles",x)['message']} 
                 for x in ['car','horse','rocket','walk']]

In [155]:
vehicle_notes

[{'name': 'car',
  'message': {'code': 230,
   'message': 'Vehicle information found.',
   'name': 'car',
   'note': 'The car is a practical powered vehicle that offers a decent compromise between speed and resource usage. It uses less fuel per move than the rocket and requires only moderate food consumption, so it can be efficient on routes that stay on solid ground. However, it cannot drive on water, and entering a water tile means the vehicle is lost immediately. That makes route safety just as important as resource planning. Fuel consumption is 0.7 per move, while food consumption is 1.0 per move.',
   'consumption': {'fuel': 0.7, 'food': 1}}},
 {'name': 'horse',
  'message': {'code': 230,
   'message': 'Vehicle information found.',
   'name': 'horse',
   'note': 'The horse is a balanced travel option for routes where you want to save fuel completely while still moving more comfortably than on foot. It does not consume any fuel at all, which makes it attractive when fuel reserves a

In [156]:
payload = {
  "apikey": api_key,
  "task": "savethem",
  "answer": ['rocket','up','right','right','up','right','up','right','right','dismount','right','right','right']
}

headers = {
    "Content-Type": "application/json"
}

try:
    response = requests.post(url_api, data=json.dumps(payload), headers=headers)
    
    # Sprawdź status odpowiedzi
    print(f"\nStatus HTTP: {response.status_code}")
    print("Odpowiedź serwera:")
    print(response.text)
    
    if response.status_code == 200:
        print("\n✅ Sukces! Dane zostały pomyślnie wysłane.")
    else:
        print(f"\n❌ Błąd. Sprawdź status HTTP: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"\n💥 Wystąpił błąd podczas wysyłania żądania: {e}")


Status HTTP: 200
Odpowiedź serwera:
{
    "code": 0,
    "message": "{FLG:INTACTCITY}"
}

✅ Sukces! Dane zostały pomyślnie wysłane.


In [157]:
print(mapa['message']['text'])

........WW
.......WW.
.T....WW..
......W...
..T...W.G.
....R.W...
...RR.WW..
SR.....W..
......WW..
.....WW...


In [162]:
payload_beavers = {
  "apikey": api_key,
  "task": "savethem",
  "answer": ['rocket','up','up','up','up','up','up','right','right','dismount','right','right','right','right','right','right','right']
}

headers = {
    "Content-Type": "application/json"
}

try:
    response = requests.post(url_api, data=json.dumps(payload_beavers), headers=headers)
    
    # Sprawdź status odpowiedzi
    print(f"\nStatus HTTP: {response.status_code}")
    print("Odpowiedź serwera:")
    print(response.text)
    
    if response.status_code == 200:
        print("\n✅ Sukces! Dane zostały pomyślnie wysłane.")
    else:
        print(f"\n❌ Błąd. Sprawdź status HTTP: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"\n💥 Wystąpił błąd podczas wysyłania żądania: {e}")


Status HTTP: 200
Odpowiedź serwera:
{
    "code": 120,
    "message": "You found beavers by the stream! {FLG:ABEAVER}"
}

✅ Sukces! Dane zostały pomyślnie wysłane.


In [160]:
gather_notes("flag")

In [161]:
gathered_notes

{'maps-endpoint': {'content': 'Terrain maps for any city can be obtained from the /api/maps endpoint. To get the correct map, send the name of the city you want to reach in the request. This makes the maps service the standard source of terrain data when planning a mission toward a specific destination. If someone needs a fresh layout for route analysis, that endpoint is where to ask first.',
  'title': 'Getting terrain maps for target cities'},
 'legend-markers': {'content': 'The map uses a compact set of symbols, and every letter matters during route planning. The letter T marks a tree tile, W marks water, and R marks rocks that block movement completely. The letter S marks the starting position, while G marks the goal that ends the mission successfully. If someone misreads even one marker, the whole path can collapse before the traveler reaches the city. Treat the legend as the first thing to decode before testing any route.',
  'title': 'Map legend and terrain markers'},
 'no-gas-s